# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/StickRift/Rift/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of analysis: Each row represents one reporting date × client × content item.

Tables used: fact_content_daily_performance, using the March and April 2026 monthly partitions.

Time window: March 1–31, 2026, provides the features available for review. April 1–30, 2026, provides the later outcome window.

Decision supported: Identify content items for human review using observed search performance. The label marks whether average daily Search Console impressions were lower in April than in March.

Deliberate exclusion: Exclude records where gsc_data_available IS NOT TRUE from analyses using Search Console metrics because those records do not have available Search Console data.

In [20]:
!pip -q install duckdb huggingface_hub pyarrow scikit-learn

from google.colab import userdata
from huggingface_hub import hf_hub_download
import duckdb
import pandas as pd
import numpy as np

# Connect to the FlyRank internship dataset
token = userdata.get("HF_TOKEN").strip()
repo_id = "FlyRank/internship-warehouse"

# Create a DuckDB connection for querying the data
con = duckdb.connect()

print("Setup complete.")
print("Dataset:", repo_id)

Setup complete.
Dataset: FlyRank/internship-warehouse


In [21]:
# Download the March and April 2026 data partitions

march_file = hf_hub_download(
    repo_id=repo_id,
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=token
)

april_file = hf_hub_download(
    repo_id=repo_id,
    filename="fact_content_daily_performance/month=2026-04/data_0.parquet",
    repo_type="dataset",
    token=token
)

print("March 2026 data downloaded successfully.")
print("April 2026 data downloaded successfully.")

March 2026 data downloaded successfully.
April 2026 data downloaded successfully.


In [22]:
# Query 1: Verify the expected grain of the March 2026 data

duplicate_check = con.execute("""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM read_parquet(?)
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 10
""", [march_file]).df()

if duplicate_check.empty:
    print("PASS: No duplicate combinations found.")
else:
    print("Duplicate combinations found:")
    display(duplicate_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

PASS: No duplicate combinations found.


In [23]:
# Query 2: Verify row count and date span for March 2026

march_summary = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT report_date) AS distinct_dates,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM read_parquet(?)
""", [march_file]).df()

display(march_summary)

,total_rows,distinct_dates,first_date,last_date
0,9841378,31,2026-03-01,2026-03-31


In [24]:
# Query 3: Check Search Console data availability

availability_check = con.execute("""
    SELECT
        COUNT(*) AS rows_before,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS rows_surviving_filter
    FROM read_parquet(?)
""", [march_file]).df()

display(availability_check)

,rows_before,rows_surviving_filter
0,9841378,3611061


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Features**

### Features

* `gsc_impressions`: Measures search visibility. Knowable at the decision moment because it is aggregated from March 2026 data.
* `gsc_clicks`: Measures clicks from search results. Knowable at the decision moment because it is aggregated from March 2026 data.
* `gsc_avg_position`: Indicates average search ranking. Knowable at the decision moment because it is aggregated from March 2026 data.
* `ga4_pageviews`: Measures page traffic. Uses March 2026 data when Google Analytics data is available; otherwise, the value may be missing.
* `ga4_engaged_sessions`: Measures engaged visits. Uses March 2026 data when Google Analytics data is available; otherwise, the value may be missing.

These features support content-review decisions using information available before the April outcome window.


These features use March 2026 data to support content-review decisions.

**Label**

`is_declining_label`: Marks whether average daily Search Console impressions were lower in April 2026 than in March 2026. This identifies pages for review; it does not prove that refreshing a page will improve performance.

**Context**

* `report_date`: Reporting date.
* `client_hash_id`: Client identifier.
* `content_hash_id`: Content item identifier.
* `gsc_data_available`: Indicates whether Search Console data is available.
* `ga4_data_available`: Indicates whether Google Analytics data is available.

**Excluded**

* Records where `gsc_data_available IS NOT TRUE` are excluded from analyses requiring Search Console metrics because search data is unavailable.
* Client and content identifiers are not predictive features.
* Future outcome data is excluded from predictive features to prevent data leakage.


In [25]:
# Build the feature vector using March data and the April outcome

feature_frame = con.execute("""
    WITH march AS (
        SELECT
            client_hash_id,
            content_hash_id,
            AVG(gsc_impressions) AS gsc_impressions,
            AVG(gsc_clicks) AS gsc_clicks,
            AVG(gsc_avg_position) AS gsc_avg_position,
            AVG(
                CASE WHEN ga4_data_available IS TRUE
                     THEN ga4_pageviews END
            ) AS ga4_pageviews,
            AVG(
                CASE WHEN ga4_data_available IS TRUE
                     THEN ga4_engaged_sessions END
            ) AS ga4_engaged_sessions,
            AVG(gsc_impressions) AS march_avg_daily_impressions
        FROM read_parquet(?)
        WHERE gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    ),
    april AS (
        SELECT
            client_hash_id,
            content_hash_id,
            AVG(gsc_impressions) AS april_avg_daily_impressions
        FROM read_parquet(?)
        WHERE gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT
        m.*,
        a.april_avg_daily_impressions,
        CASE
            WHEN a.april_avg_daily_impressions
                 < m.march_avg_daily_impressions THEN 1
            ELSE 0
        END AS is_declining_label
    FROM march m
    INNER JOIN april a
        ON m.client_hash_id = a.client_hash_id
        AND m.content_hash_id = a.content_hash_id
""", [march_file, april_file]).df()

print("Feature frame shape:", feature_frame.shape)
display(feature_frame.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (158549, 10)


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_engaged_sessions,march_avg_daily_impressions,april_avg_daily_impressions,is_declining_label
0,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,29.000000,0.032258,5.145765,NaN,NaN,29.000000,34.233333,0
1,client_62f4a7e64f5e0096,content_ac8663da7484669a,2.000000,0.000000,4.909314,NaN,NaN,2.000000,1.500000,1
2,client_62f4a7e64f5e0096,content_d49a012dcb924e31,10.612903,0.000000,5.177774,NaN,NaN,10.612903,3.892857,1
3,client_62f4a7e64f5e0096,content_614baf2af4330bd7,24.903226,0.032258,4.685335,NaN,NaN,24.903226,9.266667,1
4,client_62f4a7e64f5e0096,content_4a1ca0fa5c177e0c,1.400000,0.000000,4.266667,NaN,NaN,1.400000,1.000000,1


In [26]:
# Leakage demonstration: compare an honest model with a leaked model

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score

# Define the five features available before the prediction
feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_engaged_sessions"
]

target = "is_declining_label"

# Split the data into training and testing sets
X = feature_frame[feature_cols]
y = feature_frame[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train the honest model using only legitimate features
imputer = SimpleImputer(strategy="median")
X_train_clean = imputer.fit_transform(X_train)
X_test_clean = imputer.transform(X_test)

honest_model = DecisionTreeClassifier(max_depth=4, random_state=42)
honest_model.fit(X_train_clean, y_train)
honest_predictions = honest_model.predict(X_test_clean)

honest_accuracy = accuracy_score(y_test, honest_predictions)

# Deliberately leak the answer by including the target as an input
X_leaky = feature_frame[feature_cols + [target]]

X_train_leaky, X_test_leaky, y_train_leaky, y_test_leaky = train_test_split(
    X_leaky, y, test_size=0.2, random_state=42
)

leaky_imputer = SimpleImputer(strategy="median")
X_train_leaky_clean = leaky_imputer.fit_transform(X_train_leaky)
X_test_leaky_clean = leaky_imputer.transform(X_test_leaky)

leaky_model = DecisionTreeClassifier(max_depth=4, random_state=42)
leaky_model.fit(X_train_leaky_clean, y_train_leaky)
leaky_predictions = leaky_model.predict(X_test_leaky_clean)

leaky_accuracy = accuracy_score(y_test_leaky, leaky_predictions)

print(f"Honest model accuracy: {honest_accuracy:.4f}")
print(f"Leaked model accuracy: {leaky_accuracy:.4f}")

print("\nLEAKAGE LESSON:")
print("The leaked model can see the target label as an input.")
print("Its score is invalid because the answer is already included")
print("in the features. Never use future outcomes as model inputs.")

Honest model accuracy: 0.6793
Leaked model accuracy: 1.0000

LEAKAGE LESSON:
The leaked model can see the target label as an input.
Its score is invalid because the answer is already included
in the features. Never use future outcomes as model inputs.


In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [28]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

### Query 1 — Grain verification

**Expected grain:** One row per reporting date × client × content item.

**Result:** The duplicate check returned zero rows.

**Conclusion:** No duplicate combinations were found in the March 2026 partition.

### Query 2 — Row count and date span

**Result:** The March 2026 partition contains 9,841,378 rows across 31 distinct dates, from March 1 to March 31, 2026.

**Conclusion:** The partition covers the expected month.

### Query 3 — Search Console availability

**Result:** Of 9,841,378 March records, 3,611,061 have `gsc_data_available IS TRUE`.

**Conclusion:** Analyses using Search Console metrics must exclude records where Search Console data is unavailable.

### Feature and label verification

The feature frame contains 158,549 client-content combinations and 10 columns. March 2026 metrics are used as features, while April 2026 average daily impressions determine the outcome label.

**Conclusion:** The feature window precedes the outcome window, helping prevent future information from entering the predictive features.


In [29]:
# Section 3: Verify the data contract

# Query 1: Check for duplicate reporting-date/client/content combinations
display(duplicate_check)

if duplicate_check.empty:
    print("PASS: No duplicate combinations found in March 2026.")

# Query 2: Show the March row count and date range
display(march_summary)

# Query 3: Show Search Console data availability
display(availability_check)

# Verify the feature frame
print("Feature frame shape:", feature_frame.shape)
print("Expected: 158549 rows and 10 columns.")

,report_date,client_hash_id,content_hash_id,row_count


PASS: No duplicate combinations found in March 2026.


,total_rows,distinct_dates,first_date,last_date
0,9841378,31,2026-03-01,2026-03-31


,rows_before,rows_surviving_filter
0,9841378,3611061


Feature frame shape: (158549, 10)
Expected: 158549 rows and 10 columns.


## 4. Data limits

**Data limitations**

* **Unequal history:** Clients may have different amounts of historical data, so results may not be equally reliable across clients.
* **Data availability:** Some records have Search Console data but lack Google Analytics data. Missing Google Analytics values must not be interpreted as zero traffic.
* **Observation windows:** March is the feature window and April is the outcome window. These periods must remain separate to prevent future information from leaking into the features.
* **No causal conclusion:** Historical performance patterns can help prioritize pages for human review, but they cannot prove that refreshing a page will improve its performance.


**Data leakage demonstration:** The model using March features achieved 67.74% accuracy on the test split. The deliberately leaked model achieved 100% accuracy after the target label was included as an input. The leaked score is invalid because the model was given the answer it was supposed to predict. This demonstrates why future outcomes must be excluded from predictive features.


In [30]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:



* [x] Every section is filled with written explanations and supporting code.
* [x] The notebook runs from top to bottom without errors (Runtime → Run all).
* [x] Checked for client names and URLs; no matches were found for `client_name` or `http`. SQL queries use anonymized identifiers.
* [x] Claims use careful language about observed performance, measured outcomes, and decision support. The analysis does not claim that refreshing content causes improvement.
* [ ] Committed to my repository under `work/notebooks/` and submitted the repository URL on the assignment card.
